In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing import sequence

2026-04-08 13:11:18.133105: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775653878.329702      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775653878.382127      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775653878.826530      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775653878.826590      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775653878.826594      55 computation_placer.cc:177] computation placer alr

In [2]:

max_features = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Train size: 25000
Test size: 25000


In [3]:
max_len = 200  # max words per review

X_train = sequence.pad_sequences(X_train, maxlen=max_len)
X_test = sequence.pad_sequences(X_test, maxlen=max_len)

In [4]:
model = Sequential()

# Convert word index → dense vector
model.add(Embedding(input_dim=max_features, output_dim=128, input_length=max_len))

# LSTM layer
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

# Output layer
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-04-08 13:31:12.280512: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 97s 299ms/step - accuracy: 0.6830 - loss: 0.5741 - val_accuracy: 0.8492 - val_loss: 0.3606
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 94s 299ms/step - accuracy: 0.8449 - loss: 0.3690 - val_accuracy: 0.5542 - val_loss: 0.8416
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 94s 301ms/step - accuracy: 0.7625 - loss: 0.4780 - val_accuracy: 0.8380 - val_loss: 0.3875
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 94s 301ms/step - accuracy: 0.8847 - loss: 0.2851 - val_accuracy: 0.8472 - val_loss: 0.3595
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 94s 299ms/step - accuracy: 0.9132 - loss: 0.2238 - val_accuracy: 0.8390 - val_loss: 0.3788


In [7]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 34s 43ms/step - accuracy: 0.8408 - loss: 0.3869
Test Loss: 0.3826017677783966
Test Accuracy: 0.8424800038337708


In [8]:
word_index = imdb.get_word_index()

reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in encoded_review])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [9]:
def encode_review(text):
    words = text.lower().split()
    encoded = []

    for word in words:
        if word in word_index:
            encoded.append(word_index[word] + 3)
        else:
            encoded.append(2)  # unknown word

    return encoded

In [10]:
def predict_review(text):
    encoded = encode_review(text)
    padded = sequence.pad_sequences([encoded], maxlen=max_len)

    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        print("Positive Review 😊")
    else:
        print("Negative Review 😡")

    print("Confidence:", prediction)

In [11]:
predict_review("this movie was amazing and full of emotions")
predict_review("worst movie ever i wasted my time")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 419ms/step
Positive Review 😊
Confidence: 0.9696646
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Negative Review 😡
Confidence: 0.014859634
